This project will analyze over 900 different pickleball matches from a dataset that was recently made by scraping pklmart.com as well as DUPR.com.  However there are some issues with this dataset so cleaning will need to be done.  First, let's install our required packages.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer

# Load the dataset
df = pd.read_csv('matches_complete5.0.csv')
np.random.seed(42) #needed for later

The first issue is with the DUPR score.  The scraper that was used only scraped the based on the name of the person.  A person named John Smith may have hundreds of different DUPR profiles, making it impossible to find what the correct John Smith was.  We can solve this with Imputation based on the level of the match.  For example, if a Pro Level match contained a player that had a DUPR of 2.5, instead of discarding the data we would instead have it be the average DUPR of Pro Level matches which is 6.11.

In [ ]:
def clean_dupr_scores(df):
    level_standards = {
        'Pro': (5.0, 6.11),
        '5.0': (4.0, 5.11),
        '4.5': (3.5, 4.53),
        '4.0': (3.0, 4.11)
    }

    dupr_cols = ['TeamAPlayer1_DUPR', 'TeamAPlayer2_DUPR',
                 'TeamBPlayer1_DUPR', 'TeamBPlayer2_DUPR']

    for col in dupr_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

        def fix_score(row):
            score = row[col]
            level = row['MatchLevel']

            if level in level_standards:
                min_val, default_val = level_standards[level]
                if pd.isna(score) or score < min_val:
                    return default_val

            return score if pd.notna(score) else np.nan

        df[col] = df.apply(fix_score, axis=1)

    return df

df = clean_dupr_scores(df)

Now I want to create new columns so we can have the average DUPR of each team as well as their skill differential.

In [ ]:
df['TeamA_Avg_DUPR'] = df[['TeamAPlayer1_DUPR', 'TeamAPlayer2_DUPR']].mean(axis=1)
df['TeamB_Avg_DUPR'] = df[['TeamBPlayer1_DUPR', 'TeamBPlayer2_DUPR']].mean(axis=1)
df['DUPR_Diff'] = df['TeamA_Avg_DUPR'] - df['TeamB_Avg_DUPR']

One glaring issue with this dataset is the WinningTeam Column.  With the way the data is scraped, the Winning Team is always Team A.  This means that it is extremely easy for a machine learning model to predict which Team won.  To correct this, I will swap randomly 50% of all of the matches so that Team A wins half of all the matches, while Team B wins half of all of the matches.  

In [ ]:
swap_indices = df.sample(frac=0.5, random_state=42).index
mask = df.index.isin(swap_indices)
df.loc[mask, 'WinningTeam'] = 'TeamB'

cols = df.columns.tolist()
swap_pairs = []

for col in cols:
    if col.startswith('TeamA'):
        partner = col.replace('TeamA', 'TeamB')
        if partner in cols:
            swap_pairs.append((col, partner))

for col in cols:
    if col.startswith('Player1'):
        partner = col.replace('Player1', 'Player3')
        if partner in cols:
            swap_pairs.append((col, partner))
    elif col.startswith('Player2'):
        partner = col.replace('Player2', 'Player4')
        if partner in cols:
            swap_pairs.append((col, partner))

for col_a, col_b in swap_pairs:
    val_a = df.loc[mask, col_a].values
    val_b = df.loc[mask, col_b].values
    df.loc[mask, col_a] = val_b
    df.loc[mask, col_b] = val_a

One problem is that some columns don't have a DUPR_Diff column.  This is because the matches labeled "Other" are hard to perform imputation.  Let's remove these matches that don't have this columns.  

In [ ]:
df = df.dropna(subset=['DUPR_Diff'])

Some columns need to be dropped.  Columns like TeamA_Errors.1 are identical to TeamA_Errors.  Let's drop all columns that end in .1.  Some columns are also almost entirely empty, for example the SharpAcrossDinks columns.  The Team column is also quite redundant as it just says all of the player names.

In [ ]:
cols_to_drop = [col for col in df.columns if col.endswith('.1') or 'SharpAcross' in col]
cols_to_drop.append('Team')
df = df.drop(columns=cols_to_drop)

Let's also clean the names of these columns as well.

In [ ]:
df.columns = [c.replace('%', '_Pct').replace(' ', '_') for c in df.columns]

Instead of imputing with a zero, let's make it the average value of all of the nearest neighbors to that specific match.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
imputer = KNNImputer(n_neighbors=5, weights='distance')
df[numeric_cols] = imputer.fit_transform(df[numeric_cols])

Now this data is complete! Let's move onto the next step, exploratory data analysis!

In [ ]:
df.to_csv('matches_complete6.0.csv', index=False)